# DSC0152 - notebook v64

## Kenapa run ini berbeda

v63 melaporkan KS OOF 0.529867, tertinggi sepanjang proyek, dan public score-nya 0.557. Dua angka itu tidak blushingan: seleksi berbasis OOF mengukur noise, bukan kualitas.

Empat file punya kedua angka:

| File | KS OOF | Public |
|---|---|---|
| v57 `aggressive_145` | 0.526988 | 0.55700 |
| v58ext `ext1.5` | 0.527511 | 0.55900 |
| v61 `alt_ext0.5_tm30_lean` | 0.528624 | **0.56312** |
| v63 kandidat terbaik | **0.529867** | 0.55700 |

Pearson r pada empat pasangan itu +0.019. Pair yang menentukan: `v58ext ext1.5` dan `qc_00` punya vektor rank Spearman 0.9994, hampir identik, tapi public score berbeda 0.0027.

## Aturan yang dipatuhi

Python saja, hanya data konstruksi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Seleksi fitur dihitung per fold dari baris training fold itu saja. Factor loading dari label fold training saja. Agregat historis dipotong pada `decision_datetime`. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21,5,101'
LR = 0.01
L2 = 5.0
TOP_N = 200

# The notebook lives in agas/v64/notebook/, so the working directory is not the
# repository root and Path.cwd() is wrong here. Walk up from this file instead.
HERE = Path.cwd().resolve()
if not (HERE / 'data').is_dir():
    for cand in [HERE, *HERE.parents]:
        if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
            HERE = cand
            break
ROOT = HERE
AGAS = ROOT / 'agas'
VER = AGAS / 'v64'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)

## 1. Audit: apakah KS OOF meramal public score

Dijalankan lebih dulu, sebelum model apa pun dilatih, supaya objektif dan buktinya masuk sebelum kesimpulan.

In [ ]:
os.system(f'python3 {VER / "audit_public_vs_oof.py"}')

## 2. Seberapa besar noise sampling KS

KS adalah maksimum atas ambang dari jumlah kumulatif. Noise-nya diukur langsung dengan mengambil ulang baris dari satu vektor prediksi yang tetap, tanpa retraining sama sekali.

Tiga hasil menentukan arah run ini:

1. Peningkatan v61 ke v63 sebesar 0.001243 hanya 0.34 sampling sd.
2. Memilih argmax dari 1300 titik grid memberi kenaikan terukur sekitar 0.012 secara gratis.
3. Jarak ke 0.57 adalah 1.30 public sd, sementara SD OOF hanya 0.0036.

In [ ]:
os.system(f'python3 {VER / "diagnose_selection_noise.py"}')

## 3. Hasil negatif: seleksi stabil ditolak

Dugaan bahwa memilih kolom yang bertahan di lima probe gain independen lebih dapat diandalkan daripada satu probe. Diuji pada 10 fold dengan seed dan model yang sama persis, hanya cara seleksi yang dibedakan.

Hasilnya negatif, jadi resep v63 dipertahankan. Dicatat supaya tidak diulang.

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "check_stability_selection.py"}')

d = json.load(open(OUT / 'summary_stability_check.json'))
pd.DataFrame(d['results']).T.reset_index().rename(columns={'index': 'arm'}).sort_values('auc', ascending=False)

## 4. Model

Resep v63 dipertahankan, dengan satu perubahan: jumlah fold seed dinaikkan dari empat ke enam. Ini satu-satunya lever yang efeknya monoton di semua eksperimen, 0.845897 pada satu seed, 0.846647 pada tiga, 0.847419 dengan seleksi di atasnya.

Objektif yang dilaporkan adalah AUC. KS tetap ada di laporan, selalu bersama SD sampling-nya, tidak pernah sebagai kunci pengurutan.

In [ ]:
cmd = (f'python3 {TRAINING / "train_submission_v64.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lr {LR} --l2 {L2} --top-n {TOP_N} --out-prefix v64')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai prediksi v64 yang sudah ada')

In [ ]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

preds = OUT / 'preds_v64.npz'
if preds.exists():
    z = np.load(preds, allow_pickle=True)
    seeds = [int(s) for s in z['seeds']]
    train_ids = z['train_ids']
    apps = pd.read_csv(ROOT / 'data/train_applications.csv')
    ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
    y = ymap['target'].values.astype(int)
    rows = []
    for k in sorted(c for c in z.files if c.startswith('oof_')):
        name = k[4:]
        p = z[k]
        rows.append({'model': name, 'auc': roc_auc_score(y, p),
                     'ks': ks_2samp(p[y == 1], p[y == 0]).statistic})
    base = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
    base
else:
    print('belum ada preds_v64.npz')

## 5. Pemilihan kandidat: keragaman, bukan argmax

Builder mengumpulkan semua vektor prediksi dari v61, v62, v63, v64, membuang yang korelasinya di atas 0.998, lalu mengurutkan dengan AUC.

Alasannya: dua file yang meranking baris dengan cara yang nyaris sama memberi leaderboard draw yang sama. Mengulang ranking yang sama tidak menambah apa pun, sedangkan keragaman memberi sample tambahan dari distribusi noise. Pada 0.0053 public sd, Nilai maksimum harapan atas tiga submit diverse bernilai sekitar 0.0045, lebih besar dari seluruh peningkatan model yang berhasil dicapai di v61 sampai v63.

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "make_submission_v64.py"}')

summary = json.load(open(OUT / 'summary_v64_candidates.json'))
print('sigma OOF ', round(summary['sigma_oof'], 6))
print('sigma public', round(summary['sigma_public'], 6))
print('pool', summary['pool_size'], '-> kept', summary['kept'])
cands = pd.DataFrame(summary['candidates'])
cands

## 6. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN.

In [ ]:
ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)

rows = []
for path in sorted(OUT.glob('submission_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok})
res = pd.DataFrame(rows)
print('total', len(res), '| invalid', int((~res.ok).sum()))
res

## 7. Verifikasi semua script

Run ini mengubah cara script mengimpor `shared/paths.py`, karena folder versi ada dua kedalaman dan indeks `parents[n]` yang dipatok salah untuk salah satunya. Enam perbaikan berturutan diperlukan dan setiap tahap ditemukan lewat verifikasi. Skrip di bawah menjalankan seluruh script di `agas/` untuk memastikan `import paths` benar-benar berhasil.

In [ ]:
r = os.system(f'python3 {VER / "verify_all_scripts.py"}')
print('verify exit code', r)

## 8. Apa yang sebenarnya diketahui, dan apa yang tidak

Diketahui:

- Public score punya noise sampling sekitar 0.005 pada 30000 baris.
- Peningkatan KS OOF di bawah 0.002 tidak dapat dibedakan dari noise.
- Dua ranking dengan Spearman 0.9994 bisa berbeda 0.0027 di public.
- Marginal feature selection di 200 kolom adalah satu-satunya peningkatan model yang konsisten, sebesar 0.0023 KS OOF dan bertahan pada tiga fold seed.

Tidak diketahui:

- Apakah 0.57 bisa dicapai sama sekali. Satu-satunya titik data yang punya KS OOF 0.53 ke atas justru skornya 0.557, jadi tidak ada bukti bahwa KS OOF tinggi berkaitan dengan public tinggi.
- Berapa banyak submit slot yang tersisa. Kuota tiga per hari dengan tenggat 30 Oktober berapa banyak draw independen yang bisa diambil, dan itu jauh lebih berharga daripada optimasi model lain.

Rekomendasi: pakai slot tersisa untuk kandidat-kandidat yang benar-benar berbeda satu sama lain, bukan untuk mengirim ulang ranking yang mirip. Dan perlakukan 0.57 sebagai batas atas yang mungkin tidak terjangkau, bukan target yang bisa dikejar dengan model.